# Loading the data

In [10]:
import pandas as pd

train_data = pd.read_csv("data/train.csv")

test_data = pd.read_csv("data/test.csv")

solution = pd.read_csv("data/solution.csv")

# train_data.columns.tolist()
# test_data


train_only_cols = set(train_data.columns) - set(test_data.columns)
print(train_only_cols)

test_only_cols = set(test_data.columns)  - set(train_data.columns)
print(test_only_cols)

train_data

{'player_rating'}
set()


,Id,player_id,player_name,age,nationality,team,jersey_number,position,height_cm,weight_kg,...,defensive_contribution,possession_impact,pressure_resistance,creativity_score,consistency_score,clutch_performance_score,total_goals_tournament,total_assists_tournament,total_minutes_tournament,player_of_match_awards
0,M00001_P00053,P00053,Pedro Gavi,36,Spanish,Spain,1,Goalkeeper,191,80,...,50.8,0.0,38.6,28.6,31.4,65.4,0,0,67,0
1,M00001_P00054,P00054,Gavi Morata,27,Spanish,Spain,2,Goalkeeper,190,75,...,55.6,0.0,35.7,29.8,46.2,47.8,0,0,392,0
2,M00001_P00056,P00056,Pedro Torres,24,Spanish,Spain,4,Defender,176,78,...,70.5,6.4,52.7,31.3,68.3,59.2,0,0,324,0
3,M00001_P00058,P00058,Rodri Laporte,27,Spanish,Spain,6,Defender,188,80,...,85.6,6.4,48.7,15.3,61.8,55.1,0,1,264,0
4,M00001_P00060,P00060,Ferran Pau,35,Spanish,Spain,8,Defender,179,72,...,37.5,0.0,47.3,34.3,45.3,44.3,0,0,167,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43579,M01050_P01033,P01033,Ferland Seck,23,Senegalese,Senegal,19,Midfielder,185,79,...,38.2,3.6,49.1,53.1,44.0,60.6,0,1,162,0
43580,M01050_P01035,P01035,Sadio Diouf,24,Senegalese,Senegal,21,Forward,180,71,...,19.5,9.9,84.3,72.2,70.4,72.6,3,4,345,0
43581,M01050_P01036,P01036,Kalidou Faye,25,Senegalese,Senegal,22,Forward,165,66,...,24.7,0.0,36.4,30.8,58.3,25.1,2,1,127,0
43582,M01050_P01039,P01039,Boulaye Mendy,24,Senegalese,Senegal,25,Forward,180,77,...,21.7,5.3,74.2,76.7,67.9,38.1,1,0,276,0


As can be seen from the data, there are many irrelevant features for evaluating the player's rating. So we must choose from these features.

# Preprocessing the data

In [ ]:

columns_to_drop = ['Id', 'player_id', 'player_name', 'nationality', 'team',
       'jersey_number',
       'club_name', 'market_value_eur', 'match_id', 'match_date', 'stadium',
       'city', 'opponent_team', 'tournament_stage',
       'performance_score', 'offensive_contribution', 'defensive_contribution',
       'possession_impact', 'pressure_resistance', 'creativity_score',
       'consistency_score', 'clutch_performance_score',
       'total_goals_tournament', 'total_assists_tournament',
       'total_minutes_tournament']

X_train = train_data.drop(columns=columns_to_drop + ['player_rating'])
Y_train = train_data['player_rating']
y_true = solution['player_rating']

X_test = test_data.drop(columns=columns_to_drop)

# One hot encoding for string features
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)
# X_train

# Training the model

## Using descion tree

In [4]:
from sklearn.tree import DecisionTreeRegressor

tree_model = DecisionTreeRegressor(max_depth=5, random_state=42)

tree_model.fit(X_train, Y_train)

DecisionTreeRegressor(max_depth=5, random_state=42)

## Using XGBoost

In [ ]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=500,          # Build 500 small trees instead of 1 giant one
    learning_rate=0.05,        # Take small, careful steps to avoid overfitting
    max_depth=6,               # Let the trees grow slightly deeper to find complex rules
    colsample_bytree=0.8,      # Randomly hide 20% of columns per tree to force learning other stats
    subsample=0.9,             # Randomly use 90% of rows per tree to add variety
    random_state=42,
    n_jobs=-1                  # Use all your computer's CPU cores to train faster
)


xgb_model.fit(X_train, Y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=-1, num_parallel_tree=None, ...)

# Testing the model

In [6]:
predections_tree = tree_model.predict(X_test)

predections_xgboost = xgb_model.predict(X_test)



## Error using $R^2$

In [7]:
model_score = tree_model.score(X_test, y_true)
print(f"Descion tree Score: {model_score:.4f}")

model_score = xgb_model.score(X_test, y_true)
print(f"XGBoost Score: {model_score:.4f}")

Descion tree Score: 0.9712
XGBoost Score: 0.9741


## Error using MSE

In [8]:
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y_true, predections_tree)
print(f"Desision tree: Mean Squared Error: {mse}")

mse = mean_squared_error(y_true, predections_xgboost)
print(f"XGBoost: Mean Squared Error: {mse}")

Desision tree: Mean Squared Error: 0.2802755792529148
XGBoost: Mean Squared Error: 0.25286644241094675


# Observations

For some reason, the variance of the dataset is mostly explained by the `minutes_played` feature!

In [9]:
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': tree_model.feature_importances_
})

importance_df = importance_df.sort_values(by='Importance', ascending=False)

print("Top 10 Most Important Features:")
print(importance_df.head(10))

importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': xgb_model.feature_importances_
})

importance_df = importance_df.sort_values(by='Importance', ascending=False)

print("Top 10 Most Important Features:")
print(importance_df.head(10))

Top 10 Most Important Features:
                Feature  Importance
5        minutes_played    0.995608
14         total_passes    0.002832
6                 goals    0.000744
47     position_Forward    0.000300
39  distance_covered_km    0.000282
8                 shots    0.000181
40   sprint_distance_km    0.000041
26           recoveries    0.000013
34      save_percentage    0.000000
41        top_speed_kmh    0.000000
Top 10 Most Important Features:
                Feature  Importance
5        minutes_played    0.590029
39  distance_covered_km    0.212153
40   sprint_distance_km    0.147516
48  position_Goalkeeper    0.004826
6                 goals    0.004781
14         total_passes    0.003907
47     position_Forward    0.003850
54       match_result_W    0.002803
13    successful_passes    0.002750
7               assists    0.002109


# Saving submission

In [19]:
import numpy as np

predictions = np.clip(predections_xgboost, 1.0, 10.0)
submission = predections_xgboost.copy()

original_ids = test_data['Id'] 

# clean_ids = test_data['player_id'].str.extract(r'(\d+)')[0].astype(int)


submission = pd.DataFrame({
    'Id': original_ids,
    'player_rating': predictions
})


submission.to_csv('final_xgboost_submission.csv', index=False)